# Bletchley v1 6 Layers ONNX Model

In [6]:
# convert icev2
import torch
import pandas as pd

iceterm = "/disk/lixin/to_onnx/MMDNNv4_2/ICETerm.txt"
df = pd.read_csv(iceterm, sep="\t", header=None, names=["iceid", "new_iceid"])
df.loc[3180] = [3180, 10000]

origin_model_weight = torch.load("/disk/lixin/to_onnx/MMDNNv4_2/mm_pytorch_model.bin")
ice_embed = origin_model_weight["ice_embedding.weight"]

convert_maxtrix = torch.from_numpy(df["new_iceid"].values - 10000).to(ice_embed)
new_embed = torch.zeros([4000, 288]).to(ice_embed)
for i in range(3181):
    new_embed[int(convert_maxtrix[i]), :] = ice_embed[i, :]

origin_model_weight["ice_embedding.weight"] = new_embed
torch.save(
    origin_model_weight, "/disk/lixin/to_onnx/MMDNNv4_2/convert_mm_model_weight.bin"
)

In [2]:
import torch
import torch.nn as nn

from transformers.activations import quick_gelu
from unitorch_microsoft.pa.bletchley_v1 import (
    BletchleyTextEncoder,
    get_bletchley_text_config,
)


class MMDNNBletchleyV1ForQuery(nn.Module):
    def __init__(
        self,
        config_type: str,
        projection_dim=288,
        num_ice=4000,
        output_hidden_dim=64,
        padding_idx: int = -1,
        weight_path=None,
    ):
        super().__init__()
        config_type = get_bletchley_text_config(config_type, False)

        self.padding_idx = padding_idx
        self.text_embed_dim = config_type.hidden_size

        self.text_encoder = BletchleyTextEncoder(
            config_type, add_projection_layer=False
        )

        self.text_projection = nn.Linear(
            self.text_embed_dim,
            projection_dim,
            bias=False,
        )

        self.text_layer_norm = nn.LayerNorm(projection_dim)
        self.ice_embedding = nn.Embedding(num_ice, projection_dim)
        self.ice_layer_norm = nn.LayerNorm(projection_dim)

        self.final_text_projection = nn.Linear(
            projection_dim + projection_dim,
            output_hidden_dim,
        )

        self.from_checkpoint(weight_path)

    def from_checkpoint(self, weight_path):
        state_dict = torch.load(weight_path, map_location="cpu")
        self.load_state_dict(state_dict, strict=False)

    def forward(self, query_input_ids=None, ice_ids=None):
        query_input_ids = query_input_ids + 1
        query_input_ids = torch.cat(
            [torch.tensor([0], dtype=torch.int64).reshape(1, -1), query_input_ids],
            dim=0,
        )
        query_input_ids = torch.cat(
            [query_input_ids, torch.tensor([2], dtype=torch.int64).reshape(1, -1)],
            dim=0,
        )
        query_input_ids = query_input_ids.reshape(1, -1)
        attention_mask = torch.ones([1]).reshape(1, -1).to(query_input_ids)

        query_input_ids[query_input_ids > 250001] = 250001
        query_input_ids[query_input_ids < 0] = 0
        text_outputs = self.text_encoder(
            input_ids=query_input_ids,
            attention_mask=attention_mask,
        )
        text_embeds = text_outputs[:, 0]
        text_embeds = self.text_projection(text_embeds)

        ice_ids = ice_ids.reshape(1, -1)
        zero_tensor = torch.tensor(0).to(query_input_ids)
        ice_ids = torch.sub(ice_ids, 10000)
        ice_mask = ice_ids.ne(self.padding_idx)
        ice_ids = torch.where(ice_ids < 0, zero_tensor, ice_ids)
        ice_ids = torch.where(ice_ids > 3999, zero_tensor, ice_ids)
        ice_embeds = self.ice_embedding(ice_ids)
        ice_embeds = ice_embeds * ice_mask.unsqueeze(-1)
        if ice_embeds.dim() == 3:
            ice_embeds = ice_embeds.sum(dim=1)

        text_embeds = self.text_layer_norm(quick_gelu(text_embeds))
        ice_embeds = self.ice_layer_norm(ice_embeds)
        text_embeds = torch.cat([text_embeds, ice_embeds], dim=-1)

        text_embeds = self.final_text_projection(quick_gelu(text_embeds))
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        return text_embeds.view(1, 64), text_embeds[:, 0].view(1).squeeze()


model_file = "/disk/lixin/to_onnx/MMDNNv4_2/convert_mm_model_weight.bin"
model = MMDNNBletchleyV1ForQuery(config_type="0.3B", weight_path=model_file)
model.eval()

MMDNNBletchleyV1ForQuery(
  (text_encoder): BletchleyTextEncoder(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(128, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (postln_encoder): RobertaEncoder(
      (layer): ModuleList(
        (0): RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
        

In [3]:
query = torch.tensor([160, 13, 101089, 90]).unsqueeze(-1)
qice = torch.tensor([13000, -1, -1, -1, -1, -1, -1, -1, -1, -1]).unsqueeze(0)

torch.set_printoptions(profile="full")
torch.onnx.export(
    model,
    (query, qice),
    f="/home/lixin/model.onnx",
    input_names=["query", "qice"],
    output_names=["qvec", "dummy_score"],
    export_params=True,
    dynamic_axes={"query": {0: "querylen"}, "qice": {0: "qicelen"}},
    do_constant_folding=True,
    verbose=False,
    opset_version=13,
)

In [4]:
import torch
import onnxruntime as ort

query = torch.tensor([161, 14, 101090, 91]).unsqueeze(-1)
qice = torch.tensor(
    [13063, 13061, 12184, 12213, 12693, 13311, 10886, 11386, 13059, 13074]
).unsqueeze(0)

ort_session = ort.InferenceSession("/home/lixin/model.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "qice": qice.cpu().numpy()},
)
print("onnx output: ", outputs)
embedding, dummy_score = model(query, qice)
print(embedding)
print(dummy_score)

onnx output:  [array([[ 1.25154898e-01,  9.38191079e-03, -6.79646134e-02,
         8.61917436e-02,  4.03404236e-02, -1.61548126e-02,
        -1.14715144e-01, -3.25051956e-02,  4.44076061e-02,
        -4.39579450e-02,  5.37360832e-02,  1.21912761e-02,
         7.31123328e-01, -2.11576626e-01, -9.57470462e-02,
         2.20533856e-03,  2.76819672e-02,  1.58839762e-01,
        -1.37468517e-01,  1.07690737e-01, -3.52964103e-02,
        -2.96503827e-02,  2.23384723e-02,  2.79551931e-02,
         1.29806668e-01,  1.34985736e-02,  1.55114699e-02,
        -1.09894812e-01, -2.46457420e-02,  7.57056922e-02,
         3.41164917e-02, -2.12948620e-02, -6.38422882e-03,
        -6.20519789e-03,  5.93835749e-02,  2.52662953e-02,
        -6.00335449e-02,  2.88008004e-01,  9.47948769e-02,
         9.57531631e-02,  7.35914037e-02,  5.70216973e-04,
        -1.87655687e-02,  8.98480043e-02, -5.39248111e-03,
         4.26891074e-02, -8.53177905e-02, -2.28215158e-02,
         1.05622642e-01, -2.00772919e-02,

In [5]:
for x in ort_session.get_inputs():
    print(x)
for x in ort_session.get_outputs():
    print(x)

NodeArg(name='query', type='tensor(int64)', shape=['querylen', 1])
NodeArg(name='qice', type='tensor(int64)', shape=['qicelen', 10])
NodeArg(name='qvec', type='tensor(float)', shape=[1, 64])
NodeArg(name='dummy_score', type='tensor(float)', shape=[])


In [ ]:
import onnxruntime.transformers.optimizer as optimizer
from onnxruntime.transformers.fusion_options import FusionOptions
import onnxruntime.quantization as quantization

input_path_to_unoptimized_onnx_model = "/home/lixin/model.onnx"
output_path_to_optimized_model = "model.optimized.onnx"
output_path_to_quantized_model = "model.optimized.quantized.onnx"

optimized_model = optimizer.optimize_model(input_path_to_unoptimized_onnx_model)
optimized_model.save_model_to_file(output_path_to_optimized_model)

quantization.quantize_dynamic(
    output_path_to_optimized_model, output_path_to_quantized_model
)

# Bletchley v1 3 Layers ONNX Model

In [7]:
import torch
import torch.nn as nn

from transformers.activations import quick_gelu
from unitorch_microsoft.pa.bletchley_v1 import (
    BletchleyTextEncoder,
    get_bletchley_text_config,
)


class MMDNNBletchleyV1Layers3ForQuery(nn.Module):
    def __init__(
        self,
        config_type: str,
        num_query_layers=3,
        projection_dim=288,
        num_ice=4000,
        output_hidden_dim=64,
        padding_idx: int = -1,
        weight_path=None,
    ):
        super().__init__()
        config_type = get_bletchley_text_config(config_type, False)

        self.padding_idx = padding_idx
        self.text_embed_dim = config_type.hidden_size
        config_type.num_hidden_layers = num_query_layers

        self.text_encoder = BletchleyTextEncoder(
            config_type, add_projection_layer=False
        )

        self.text_projection = nn.Linear(
            self.text_embed_dim,
            projection_dim,
            bias=False,
        )

        self.text_layer_norm = nn.LayerNorm(projection_dim)
        self.ice_embedding = nn.Embedding(num_ice, projection_dim)
        self.ice_layer_norm = nn.LayerNorm(projection_dim)

        self.final_text_projection = nn.Linear(
            projection_dim + projection_dim,
            output_hidden_dim,
        )

        # self.from_checkpoint(weight_path)

    def from_checkpoint(self, weight_path):
        state_dict = torch.load(weight_path, map_location="cpu")
        self.load_state_dict(state_dict, strict=False)

    def forward(self, query_input_ids=None, ice_ids=None):
        query_input_ids = query_input_ids + 1
        query_input_ids = torch.cat(
            [torch.tensor([0], dtype=torch.int64).reshape(1, -1), query_input_ids],
            dim=0,
        )
        query_input_ids = torch.cat(
            [query_input_ids, torch.tensor([2], dtype=torch.int64).reshape(1, -1)],
            dim=0,
        )
        query_input_ids = query_input_ids.reshape(1, -1)
        attention_mask = torch.ones([1]).reshape(1, -1).to(query_input_ids)

        query_input_ids[query_input_ids > 250001] = 250001
        query_input_ids[query_input_ids < 0] = 0
        text_outputs = self.text_encoder(
            input_ids=query_input_ids,
            attention_mask=attention_mask,
        )
        text_embeds = text_outputs[:, 0]
        text_embeds = self.text_projection(text_embeds)

        ice_ids = ice_ids.reshape(1, -1)
        zero_tensor = torch.tensor(0).to(query_input_ids)
        ice_ids = torch.sub(ice_ids, 10000)
        ice_mask = ice_ids.ne(self.padding_idx)
        ice_ids = torch.where(ice_ids < 0, zero_tensor, ice_ids)
        ice_ids = torch.where(ice_ids > 3999, zero_tensor, ice_ids)
        ice_embeds = self.ice_embedding(ice_ids)
        ice_embeds = ice_embeds * ice_mask.unsqueeze(-1)
        if ice_embeds.dim() == 3:
            ice_embeds = ice_embeds.sum(dim=1)

        text_embeds = self.text_layer_norm(quick_gelu(text_embeds))
        ice_embeds = self.ice_layer_norm(ice_embeds)
        text_embeds = torch.cat([text_embeds, ice_embeds], dim=-1)

        text_embeds = self.final_text_projection(quick_gelu(text_embeds))
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        return text_embeds.view(1, 64), text_embeds[:, 0].view(1).squeeze()


model_file = "/disk/lixin/to_onnx/MMDNNv4_2/convert_mm_model_weight.bin"
model = MMDNNBletchleyV1Layers3ForQuery(config_type="0.3B", weight_path=model_file)
model.eval()

MMDNNBletchleyV1Layers3ForQuery(
  (text_encoder): BletchleyTextEncoder(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(128, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (postln_encoder): RobertaEncoder(
      (layer): ModuleList(
        (0): RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
 

In [8]:
query = torch.tensor([160, 13, 101089, 90]).unsqueeze(-1)
qice = torch.tensor([13000, -1, -1, -1, -1, -1, -1, -1, -1, -1]).unsqueeze(0)

torch.set_printoptions(profile="full")
torch.onnx.export(
    model,
    (query, qice),
    f="/home/lixin/model_3layer.onnx",
    input_names=["query", "qice"],
    output_names=["qvec", "dummy_score"],
    export_params=True,
    dynamic_axes={"query": {0: "querylen"}},
    do_constant_folding=True,
    verbose=False,
    opset_version=13,
)

In [9]:
import torch
import onnxruntime as ort

query = torch.tensor([161, 14, 101090, 91]).unsqueeze(-1)
qice = torch.tensor(
    [13063, 13061, 12184, 12213, 12693, 13311, 10886, 11386, 13059, 13074]
).unsqueeze(0)

ort_session = ort.InferenceSession("/home/lixin/model_3layer.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "qice": qice.cpu().numpy()},
)
print("onnx output: ", outputs)
embedding, dummy_score = model(query, qice)
print(embedding)
print(dummy_score)

onnx output:  [array([[-0.03844412,  0.14782088, -0.05358408,  0.01034466,  0.07681898,
         0.05012779,  0.1496336 ,  0.14447908,  0.14068449, -0.3757385 ,
        -0.05264235,  0.07091296,  0.04213703, -0.04530224, -0.19900453,
         0.05116792, -0.08354289, -0.22946548,  0.05997501, -0.08320741,
         0.04367742,  0.07872839,  0.02427922, -0.108702  ,  0.18805672,
         0.12869757, -0.08854228, -0.08836111, -0.13628425, -0.19002323,
         0.11061412, -0.12427342,  0.01281217, -0.02837195, -0.11664154,
         0.04923155,  0.04474371,  0.07955455, -0.12879673, -0.12366925,
         0.00116744, -0.01821342,  0.28562865, -0.13507879, -0.1767855 ,
         0.16703066, -0.10433212, -0.06636357, -0.15289171, -0.05924522,
        -0.12312573,  0.15239652,  0.02285444, -0.12702066, -0.09485704,
         0.14009348, -0.1336233 ,  0.08740682, -0.18120845, -0.07735348,
         0.20124038,  0.12299481, -0.05242557, -0.03916952]],
      dtype=float32), array(-0.03844412, dtype=

In [10]:
for x in ort_session.get_inputs():
    print(x)
for x in ort_session.get_outputs():
    print(x)

NodeArg(name='query', type='tensor(int64)', shape=['querylen', 1])
NodeArg(name='qice', type='tensor(int64)', shape=['qicelen', 10])
NodeArg(name='qvec', type='tensor(float)', shape=[1, 64])
NodeArg(name='dummy_score', type='tensor(float)', shape=[])
